# 🔍 04_infer_pi.ipynb — TFLite Inference on Raspberry Pi 4
**Run on Raspberry Pi 4.**

يستخدم أحسن vision model من **01_train_vision.ipynb** (`models/best_vision.tflite`).
يلتقط صورة → يصنف → يبعت serial command للـ Arduino.

- Press **ENTER** للـ capture
- اكتب `q` للخروج

## Cell 1 — Imports

In [1]:
import json, time, os
import numpy as np
import cv2
import serial

try:
    import tflite_runtime.interpreter as tflite
    print('tflite_runtime ✅  Pi mode')
except ImportError:
    import tensorflow.lite as tflite
    print('tensorflow.lite ✅  PC mode')

print('OpenCV :', cv2.__version__)

tensorflow.lite ✅  PC mode
OpenCV : 4.11.0


## Cell 2 — Config

In [2]:
# ── Vision ────────────────────────────────────────────────────────────────────
TFLITE_MODEL_PATH    = 'models/best_vision.tflite'   # من 01_train_vision
LABELS_PATH          = 'models/labels.json'
IMAGE_SIZE           = (224, 224)
CONFIDENCE_THRESHOLD = 0.80
CAM_INDEX            = 0

# ── Serial ────────────────────────────────────────────────────────────────────
SERIAL_PORT     = '/dev/ttyUSB0'
SERIAL_BAUDRATE = 9600
SERIAL_TIMEOUT  = 2

SERIAL_COMMANDS = {
    'ketofan'   : 'A',
    'brufen'    : 'B',
    'cataflam'  : 'C',
    'ambezim'   : 'D',
    'trimed_flu': 'A',   # ⚠️ مفيش station خاص — بيروح A مؤقتاً,
    'sleep'     : 'S',   # S = line tracking / home,
}

INVENTORY_PATH = 'inventory.json'

print('Config ✅')

Config ✅


## Cell 3 — Load Model & Labels

In [3]:
interpreter = tflite.Interpreter(model_path=TFLITE_MODEL_PATH, num_threads=4)
interpreter.allocate_tensors()
_inp = interpreter.get_input_details()
_out = interpreter.get_output_details()

with open(LABELS_PATH) as f:
    labels = {int(k): v for k, v in json.load(f).items()}

# Preprocess type من 01 meta
try:
    with open('models/best_vision_meta.json') as f:
        vision_meta = json.load(f)
    PREPROCESS_TYPE = vision_meta.get('preprocess_type', 'mobilenet')
    print(f'Model       : {vision_meta["best_model"]}  (acc={vision_meta["best_acc"]:.4f})')
except FileNotFoundError:
    PREPROCESS_TYPE = 'mobilenet'
    print('Model loaded (no meta, assuming mobilenet)')

print(f'Input shape : {_inp[0]["shape"]}')
print(f'Labels      : {labels}')

Model       : MobileNetV2  (acc=1.0000)
Input shape : [  1 224 224   3]
Labels      : {0: 'ambezim', 1: 'brufen', 2: 'cataflam', 3: 'ketofan', 4: 'trimed_flu'}


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## Cell 4 — Serial Port

In [4]:
try:
    ser = serial.Serial(SERIAL_PORT, SERIAL_BAUDRATE, timeout=SERIAL_TIMEOUT)
    time.sleep(2)
    print(f'Serial ✅  {SERIAL_PORT}')
except serial.SerialException as e:
    ser = None
    print(f'Serial ⚠️  {e}  — running without Arduino')

Serial ⚠️  [Errno 2] could not open port /dev/ttyUSB0: [Errno 2] No such file or directory: '/dev/ttyUSB0'  — running without Arduino


## Cell 5 — Helper Functions

In [5]:
def preprocess(frame: np.ndarray) -> np.ndarray:
    h, w = IMAGE_SIZE
    img  = cv2.resize(frame, (w, h))
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    if PREPROCESS_TYPE == 'efficientnet':
        img = (img / 127.5) - 1.0
    else:
        img = img / 255.0
    return np.expand_dims(img, axis=0)


def classify(frame: np.ndarray):
    """Returns (label, confidence) or (None, conf)."""
    img = preprocess(frame)
    interpreter.set_tensor(_inp[0]['index'], img)
    interpreter.invoke()
    probs    = interpreter.get_tensor(_out[0]['index'])[0]
    top_idx  = int(np.argmax(probs))
    top_conf = float(probs[top_idx])
    label    = labels[top_idx]

    print('\n── Result ─────────────────────────────────')
    for idx in sorted(labels):
        bar = '█' * int(probs[idx] * 20)
        mrk = ' ◄' if idx == top_idx else ''
        print(f'  {labels[idx]:<14} {probs[idx]:.2%}  {bar}{mrk}')

    return (label, top_conf) if top_conf >= CONFIDENCE_THRESHOLD else (None, top_conf)


def send_serial(label: str):
    # 1. تنظيف الاسم (تحويل لـ lowercase وتبديل المسافات بـ underscore)
    # ده بيضمن إن "Trimed Flu" تتحول لـ "trimed_flu" فتلاقي حرف "S" في القاموس
    clean_label = label.lower().strip().replace(" ", "_")
    
    # 2. جلب الحرف المقابل من القاموس (A, B, C, D, S)
    cmd = SERIAL_COMMANDS.get(clean_label, '?')
    
    print(f'[📡 Serial] → Sending: {repr(cmd)} for ({label})')
    
    if ser and ser.is_open:
        ser.write(cmd.encode())
        time.sleep(0.1)
        if ser.in_waiting:
            try:
                resp = ser.readline().decode().strip()
                print(f'[📡 Serial] ← Received: {resp!r}')
            except:
                pass


def load_inventory():
    if not os.path.exists(INVENTORY_PATH):
        return {k: 10 for k in labels.values()}
    with open(INVENTORY_PATH) as f:
        return json.load(f)

def save_inventory(inv):
    with open(INVENTORY_PATH, 'w') as f:
        json.dump(inv, f, indent=2)

def decrement_inventory(medicine):
    inv = load_inventory()
    qty = inv.get(medicine, 0)
    if qty > 0:
        inv[medicine] = qty - 1
        save_inventory(inv)
        return qty - 1
    return 0

print('Helpers ✅')

Helpers ✅


## Cell 6 — 🚀 Manual Inference Loop (ENTER to capture)

In [ ]:
cap = cv2.VideoCapture(CAM_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    raise RuntimeError(f'Cannot open camera index {CAM_INDEX}')

print('=' * 44)
print('  Press ENTER to capture — type q to quit')
print('=' * 44)

scan_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        print('Camera read failed'); break

    # Overlay
    cv2.putText(frame, 'ENTER=Scan  q=Quit', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
    cv2.imshow('Therapeía — Pi Inference', frame)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        print('Quitting...')
        break
    elif key == 13:  # ENTER
        scan_count += 1
        print(f'\n[Scan #{scan_count}]')
        t0            = time.time()
        label, conf   = classify(frame)
        elapsed       = time.time() - t0

        if label:
            remaining = decrement_inventory(label)
            send_serial(label)
            print(f'  ✅ {label} — {conf:.2%}  ({elapsed*1000:.0f}ms)  '
                  f'remaining: {remaining}')
        else:
            print(f'  ⚠️  Low confidence ({conf:.2%}) — please retry')

cap.release()
cv2.destroyAllWindows()
if ser and ser.is_open:
    ser.write(b'H\n')  # home position
    ser.close()
print('Done.')

  Press ENTER to capture — type q to quit
Quitting...
Done.


: 